# Chapter 3
## Essential Operations

This is a secondary notebook dedicated mostly to using the DeltaTable API.

This depends on previously running the first chapter 3 notebook.

In [7]:
# Databricks Free Edition: Spark runs on serverless via Databricks Connect (see dldg.py at the repo root)
from dldg import spark, DATASETS, WORK, set_conf, tree, rm, upload_dir
CH = f"{WORK}/ch03"  # replaces the Docker image's /opt/spark/work-dir

In [8]:
# Create a DeltaTable object to interact with
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, f"{CH}/countries.delta")
delta_table.toDF()


DataFrame[id: bigint, country: string, capital: string]

In [9]:
# View the table history
delta_table.history().show()

+-------+-------------------+--------------+--------------------+------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|        userId|            userName|   operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+--------------+--------------------+------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      9|2026-10-23 23:04:36|74385489255950|shridhar.upadhyay...|       MERGE|{predicate -> ["(...|NULL|              NULL|   b8b39f10-2516-467...|1023-230255-h3pa9...|          8|WriteSerializabl

In [10]:
# With a specific timestamp you can restore time specific versions
# delta_table.restoreToTimestamp('2024-05-24').show()

In [11]:
# You can also restore by version number (easier in this case since we just made the tables)
# restoreToVersion() fails over Databricks Connect (CANNOT_INFER_EMPTY_SCHEMA); the SQL equivalent works
spark.sql(f"RESTORE TABLE delta.`{CH}/countries.delta` TO VERSION AS OF 1").show()
# delta_table.restoreToVersion(1)

+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|table_size_after_restore|num_of_files_after_restore|num_removed_files|num_restored_files|removed_files_size|restored_files_size|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|                    1335|                         1|                3|                 1|              3877|               1335|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+



In [17]:
# With Pyspark you can use time travel to read from the table, update the value accordingly
# Use the timestamp of version 1 from the history, since a fixed date is outside your table's history
# (cast to string on Databricks, so it stays in the server's timezone rather than your local one)
ts = delta_table.history().where("version = 1").selectExpr("cast(timestamp as string)").first()[0]
spark.read.option("timestampAsOf", ts).load(f"{CH}/countries.delta").count()

2

In [19]:
# Pyspark time travel by version number
spark.read.option("versionAsOf", "1").load(f"{CH}/countries.delta").select("id").distinct().show()

+---+
| id|
+---+
|  1|
|  2|
+---+



In [20]:
# Delteing values with Pyspark functions
from pyspark.sql.functions import col

delta_table.delete(col("id") == 1)
df = delta_table.toDF()
df.show()

+---+-------+-------+
| id|country|capital|
+---+-------+-------+
|  2| Canada|Toronto|
+---+-------+-------+



In [21]:
# permissions error? works correctly though
(
spark
.createDataFrame(
    [
    (1, 'India', 'New Delhi'),
    (4, 'Australia', 'Canberra')
    ],
    schema=["id", "country", "capital"]
    )
.write
.format("delta")
.mode("overwrite")
.save(f"{CH}/exampleDB.delta")
)

In [16]:
# not working right with the derby metastore but if you need to replace with the DeltaTable API this is how it is done
# delta_table2 = (
#     DeltaTable.replace(spark)
#     .tableName("countries.delta")
#     .addColumns(df.schema)
#     .execute()
#     )

In [22]:
# Using the mergeBuilder to upsert values
idf = (
    spark
    .createDataFrame([
        (1, 'India', 'New Delhi'),
        (4, 'Australia', 'Canberra')],
        schema=["id", "country", "capital"]
        )
    )

delta_table.alias("target").merge(
    source = idf.alias("source"),
    condition = "source.id = target.id"
  ).whenMatchedUpdate(set =
    {
      "country": "source.country",
      "capital": "source.capital"
    }
  ).whenNotMatchedInsert(values =
    {
      "id": "source.id",
      "country": "source.country",
      "capital": "source.capital"
    }
  ).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [23]:
# Check the final results
delta_table.toDF().show()

+---+---------+---------+
| id|  country|  capital|
+---+---------+---------+
|  4|Australia| Canberra|
|  1|    India|New Delhi|
|  2|   Canada|  Toronto|
+---+---------+---------+

